# Modelling the row decision

## Six outcomes, five shapes
Every engine classifies each fuel row into exactly one of six outcomes:
`Accepted`, `AcceptedWithWarnings`, `Quarantined`, `SkippedDuplicate`,
`Rejected`, `Fatal`. The decision is the **central type** of the domain
— almost every other type either feeds it or is computed from it.

What changes across the five engines is **how that "exactly one of"
shows up in the type system**. The shape of the type is the shape of
the answer to the question "what can this value be?"

## Normal C# — one flat class with a string discriminator

In [ ]:
// normalcsharp-fuel-engine/src/FuelUploadEngine/Models/RowDecision.cs
public class RowDecision
{
    public int RowNumber;
    public string Status;                   // "Accepted" | "AcceptedWithWarnings" | "Quarantined" | "Skipped" | "Rejected" | "Fatal"
    public FuelTransaction Transaction;     // null when Status is Rejected / Fatal / Skipped
    public Vehicle Vehicle;                 // null when vehicle lookup failed
    public List<string> Errors = new List<string>();
    public List<string> Warnings = new List<string>();
    public List<string> QuarantineReasons = new List<string>();
    public string SkipReason;
    public string FatalMessage;
}

Every field exists on every decision. Which fields are *meaningful*
depends on which string `Status` carries. The compiler can't help —
it sees a flat record where any combination of populated and null
fields is structurally legal. "Accepted with `FatalMessage` populated"
is a value this type happily holds.

Cost: every consumer must defensively null-check, switch on a string,
and trust the producer's invariants. The comment in the source
("populated when Status == ...") *is* the type system here.

## Idiomatic C# — sealed-record discriminated union

In [ ]:
// csharp-fuel-engine/src/FuelUploadEngine/Domain/RowDecision.cs
public abstract record RowDecision
{
    private RowDecision() { }

    public sealed record AcceptedTransaction(FuelTransaction Transaction) : RowDecision;
    public sealed record AcceptedTransactionWithWarnings(
        FuelTransaction Transaction,
        IReadOnlyList<UploadWarning> Warnings) : RowDecision;
    public sealed record QuarantinedRow : RowDecision { /* checked ctor */ }
    public sealed record SkippedDuplicate(
        RowNumber RowNumber, DuplicateState Duplicate, DuplicateSkipCode Reason) : RowDecision;
    public sealed record RejectedRow(RowNumber RowNumber, RejectionReason Reason) : RowDecision;
    public sealed record FatalProcessingError(RowNumber RowNumber, FatalError Error) : RowDecision;
}

The private constructor on the abstract parent is load-bearing — it
prevents anyone outside the file declaring a seventh subtype. Each
case carries exactly the fields it needs and nothing else. The
`QuarantinedRow` constructor checks `reasons.Count > 0` at runtime —
the closest C# can get to "non-empty list" at the type level.

Cost: a lot of boilerplate, and consumers still rely on the **switch
expression** to be exhaustive. There is no compile-time exhaustiveness
guarantee — see [exhaustiveness](r5-exhaustiveness.ipynb) for what that
means in practice.

## F# — native discriminated union

In [ ]:
// fsharp-fuel-engine/FuelUpload.Domain/Decision.fs
[<RequireQualifiedAccess>]
type RowDecision =
    | Accepted of AcceptedTransaction
    | AcceptedWithWarnings of AcceptedTransaction * Warning list
    | Quarantined of QuarantinedRow
    | SkippedDuplicate of SkippedDuplicate
    | Rejected of RejectedRow
    | Fatal of FatalProcessingError

Six lines. The compiler enforces that you can only construct one of
these six things. Pattern matching is enforced exhaustive by default
(with a configurable warning that you can promote to an error). The
non-empty list for quarantine reasons is enforced by a separate
`QuarantineReasons` wrapper type with a private constructor —
declared once, used everywhere.

Cost: F# is still on .NET. The CLR sees these as classes and pattern
matching compiles to runtime type tests. The `[<CLIMutable>]`
attribute on adjacent records (used for JSON interop) opens a null
hole for the unwary — see [null](r7-null.ipynb).

## Haskell — algebraic data type

In [ ]:
-- haskell-fuel-engine/src/FuelUpload/Domain/Decision.hs
data RowDecision
  = Accepted FuelTransaction
  | AcceptedWithWarnings FuelTransaction (NonEmpty ValidationWarning)
  | Quarantined FuelTransaction (NonEmpty QuarantineReason)
  | SkippedDuplicate SkippedDuplicate
  | Rejected RejectedRow
  | Fatal FatalError
  deriving stock (Eq, Show)

Same shape as F#, with one quiet upgrade: `NonEmpty` is a real type
constructor, not a wrapper around a list with a runtime check. A
function returning `NonEmpty QuarantineReason` *cannot* return an
empty list — you cannot construct one. The "no quarantined row
without reasons" invariant is enforced at compile time, statically,
with no boilerplate.

Cost: it's Haskell. The runtime story is non-trivial to integrate
into a .NET shop, and `NonEmpty` is one example of a library type
that doesn't have a frictionless C# equivalent. See [the
boundary](r4-boundary.ipynb).

## Rust — sum type via `enum`

In [ ]:
// rust-fuel-engine/src/domain/decision.rs
pub enum RowDecision {
    Accepted(FuelTransaction),
    Warning {
        transaction: FuelTransaction,
        warnings: Vec<Warning>,
    },
    Quarantined {
        transaction: FuelTransaction,
        reasons: QuarantineReasons,
        warnings: Vec<Warning>,
    },
    SkippedDuplicate(SkippedDuplicate),
    Rejected(RejectedRow),
    Fatal(FatalError),
}

Rust's `enum` is a proper tagged union. Each variant can have its
own positional or named fields. `match` on a `RowDecision` is
**exhaustive at compile time** — leaving a variant out is a compiler
error, not a warning. `QuarantineReasons` is a newtype that wraps a
`Vec` and refuses to be constructed empty — same trick as F#'s
non-empty wrapper.

Cost: ownership and lifetimes. The cases themselves are cheap, but
moving them around — especially across a borrow boundary into a DTO
— costs you the borrow checker's full attention.

## The ladder, summarised

<div class="mermaid-diagram" style="width:100%">
<svg id="mermaid-figure-1" xmlns="http://www.w3.org/2000/svg" xlink="http://www.w3.org/1999/xlink" class="flowchart" viewbox="0 0 886.234375 222" role="graphics-document document" aria-roledescription="flowchart-v2" style="width:100%;height:auto;max-width:100%;display:block">
<style>#mermaid-figure-1{font-family:"trebuchet ms",verdana,arial,sans-serif;font-size:16px;fill:#000000;}@keyframes edge-animation-frame{from{stroke-dashoffset:0;}}@keyframes dash{to{stroke-dashoffset:0;}}#mermaid-figure-1 .edge-animation-slow{stroke-dasharray:9,5!important;stroke-dashoffset:900;animation:dash 50s linear infinite;stroke-linecap:round;}#mermaid-figure-1 .edge-animation-fast{stroke-dasharray:9,5!important;stroke-dashoffset:900;animation:dash 20s linear infinite;stroke-linecap:round;}#mermaid-figure-1 .error-icon{fill:#552222;}#mermaid-figure-1 .error-text{fill:#552222;stroke:#552222;}#mermaid-figure-1 .edge-thickness-normal{stroke-width:1px;}#mermaid-figure-1 .edge-thickness-thick{stroke-width:3.5px;}#mermaid-figure-1 .edge-pattern-solid{stroke-dasharray:0;}#mermaid-figure-1 .edge-thickness-invisible{stroke-width:0;fill:none;}#mermaid-figure-1 .edge-pattern-dashed{stroke-dasharray:3;}#mermaid-figure-1 .edge-pattern-dotted{stroke-dasharray:2;}#mermaid-figure-1 .marker{fill:#666;stroke:#666;}#mermaid-figure-1 .marker.cross{stroke:#666;}#mermaid-figure-1 svg{font-family:"trebuchet ms",verdana,arial,sans-serif;font-size:16px;}#mermaid-figure-1 p{margin:0;}#mermaid-figure-1 .label{font-family:"trebuchet ms",verdana,arial,sans-serif;color:#000000;}#mermaid-figure-1 .cluster-label text{fill:#333;}#mermaid-figure-1 .cluster-label span{color:#333;}#mermaid-figure-1 .cluster-label span p{background-color:transparent;}#mermaid-figure-1 .label text,#mermaid-figure-1 span{fill:#000000;color:#000000;}#mermaid-figure-1 .node rect,#mermaid-figure-1 .node circle,#mermaid-figure-1 .node ellipse,#mermaid-figure-1 .node polygon,#mermaid-figure-1 .node path{fill:#eee;stroke:#999;stroke-width:1px;}#mermaid-figure-1 .rough-node .label text,#mermaid-figure-1 .node .label text,#mermaid-figure-1 .image-shape .label,#mermaid-figure-1 .icon-shape .label{text-anchor:middle;}#mermaid-figure-1 .node .katex path{fill:#000;stroke:#000;stroke-width:1px;}#mermaid-figure-1 .rough-node .label,#mermaid-figure-1 .node .label,#mermaid-figure-1 .image-shape .label,#mermaid-figure-1 .icon-shape .label{text-align:center;}#mermaid-figure-1 .node.clickable{cursor:pointer;}#mermaid-figure-1 .root .anchor path{fill:#666!important;stroke-width:0;stroke:#666;}#mermaid-figure-1 .arrowheadPath{fill:#333333;}#mermaid-figure-1 .edgePath .path{stroke:#666;stroke-width:2.0px;}#mermaid-figure-1 .flowchart-link{stroke:#666;fill:none;}#mermaid-figure-1 .edgeLabel{background-color:white;text-align:center;}#mermaid-figure-1 .edgeLabel p{background-color:white;}#mermaid-figure-1 .edgeLabel rect{opacity:0.5;background-color:white;fill:white;}#mermaid-figure-1 .labelBkg{background-color:rgba(255, 255, 255, 0.5);}#mermaid-figure-1 .cluster rect{fill:hsl(0, 0%, 98.9215686275%);stroke:#707070;stroke-width:1px;}#mermaid-figure-1 .cluster text{fill:#333;}#mermaid-figure-1 .cluster span{color:#333;}#mermaid-figure-1 div.mermaidTooltip{position:absolute;text-align:center;max-width:200px;padding:2px;font-family:"trebuchet ms",verdana,arial,sans-serif;font-size:12px;background:hsl(-160, 0%, 93.3333333333%);border:1px solid #707070;border-radius:2px;pointer-events:none;z-index:100;}#mermaid-figure-1 .flowchartTitleText{text-anchor:middle;font-size:18px;fill:#000000;}#mermaid-figure-1 rect.text{fill:none;stroke-width:0;}#mermaid-figure-1 .icon-shape,#mermaid-figure-1 .image-shape{background-color:white;text-align:center;}#mermaid-figure-1 .icon-shape p,#mermaid-figure-1 .image-shape p{background-color:white;padding:2px;}#mermaid-figure-1 .icon-shape rect,#mermaid-figure-1 .image-shape rect{opacity:0.5;background-color:white;fill:white;}#mermaid-figure-1 .label-icon{display:inline-block;height:1em;overflow:visible;vertical-align:-0.125em;}#mermaid-figure-1 .node .label-icon path{fill:currentColor;stroke:revert;stroke-width:revert;}#mermaid-figure-1 :root{--mermaid-font-family:"trebuchet ms",verdana,arial,sans-serif;}</style>
<g><marker id="mermaid-1779381586056_flowchart-v2-pointEnd" class="marker flowchart-v2" viewbox="0 0 10 10" refx="5" refy="5" markerunits="userSpaceOnUse" markerwidth="8" markerheight="8" orient="auto"><path d="M 0 0 L 10 5 L 0 10 z" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></path></marker><marker id="mermaid-1779381586056_flowchart-v2-pointStart" class="marker flowchart-v2" viewbox="0 0 10 10" refx="4.5" refy="5" markerunits="userSpaceOnUse" markerwidth="8" markerheight="8" orient="auto"><path d="M 0 5 L 10 10 L 10 0 z" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></path></marker><marker id="mermaid-1779381586056_flowchart-v2-circleEnd" class="marker flowchart-v2" viewbox="0 0 10 10" refx="11" refy="5" markerunits="userSpaceOnUse" markerwidth="11" markerheight="11" orient="auto"><circle cx="5" cy="5" r="5" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></circle></marker><marker id="mermaid-1779381586056_flowchart-v2-circleStart" class="marker flowchart-v2" viewbox="0 0 10 10" refx="-1" refy="5" markerunits="userSpaceOnUse" markerwidth="11" markerheight="11" orient="auto"><circle cx="5" cy="5" r="5" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></circle></marker><marker id="mermaid-1779381586056_flowchart-v2-crossEnd" class="marker cross flowchart-v2" viewbox="0 0 11 11" refx="12" refy="5.2" markerunits="userSpaceOnUse" markerwidth="11" markerheight="11" orient="auto"><path d="M 1,1 l 9,9 M 10,1 l -9,9" class="arrowMarkerPath" style="stroke-width: 2; stroke-dasharray: 1, 0;"></path></marker><marker id="mermaid-1779381586056_flowchart-v2-crossStart" class="marker cross flowchart-v2" viewbox="0 0 11 11" refx="-1" refy="5.2" markerunits="userSpaceOnUse" markerwidth="11" markerheight="11" orient="auto"><path d="M 1,1 l 9,9 M 10,1 l -9,9" class="arrowMarkerPath" style="stroke-width: 2; stroke-dasharray: 1, 0;"></path></marker><g class="root"><g class="clusters"></g><g class="edgePaths"><path d="M166.266,111L170.432,111C174.599,111,182.932,111,190.599,111C198.266,111,205.266,111,208.766,111L212.266,111" id="L_A_B_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_A_B_0" data-points="W3sieCI6MTY2LjI2NTYyNSwieSI6MTExfSx7IngiOjE5MS4yNjU2MjUsInkiOjExMX0seyJ4IjoyMTYuMjY1NjI1LCJ5IjoxMTF9XQ==" marker-end="url(#mermaid-1779381586056_flowchart-v2-pointEnd)"></path><path d="M381.219,111L385.385,111C389.552,111,397.885,111,405.552,111C413.219,111,420.219,111,423.719,111L427.219,111" id="L_B_C_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_B_C_0" data-points="W3sieCI6MzgxLjIxODc1LCJ5IjoxMTF9LHsieCI6NDA2LjIxODc1LCJ5IjoxMTF9LHsieCI6NDMxLjIxODc1LCJ5IjoxMTF9XQ==" marker-end="url(#mermaid-1779381586056_flowchart-v2-pointEnd)"></path><path d="M612.106,72L620.434,67.833C628.763,63.667,645.421,55.333,657.249,51.167C669.078,47,676.078,47,679.578,47L683.078,47" id="L_C_D_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_C_D_0" data-points="W3sieCI6NjEyLjEwNTU5MDgyMDMxMjUsInkiOjcyfSx7IngiOjY2Mi4wNzgxMjUsInkiOjQ3fSx7IngiOjY4Ny4wNzgxMjUsInkiOjQ3fV0=" marker-end="url(#mermaid-1779381586056_flowchart-v2-pointEnd)"></path><path d="M612.106,150L620.434,154.167C628.763,158.333,645.421,166.667,659.693,170.833C673.966,175,685.854,175,691.798,175L697.742,175" id="L_C_E_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_C_E_0" data-points="W3sieCI6NjEyLjEwNTU5MDgyMDMxMjUsInkiOjE1MH0seyJ4Ijo2NjIuMDc4MTI1LCJ5IjoxNzV9LHsieCI6NzAxLjc0MjE4NzUsInkiOjE3NX1d" marker-end="url(#mermaid-1779381586056_flowchart-v2-pointEnd)"></path></g><g class="edgeLabels"><g class="edgeLabel"><g class="label" data-id="L_A_B_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_B_C_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_C_D_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_C_E_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g></g><g class="nodes"><g class="node default" id="flowchart-A-0" transform="translate(87.1328125, 111)"><rect class="basic label-container" style="" x="-79.1328125" y="-39" width="158.265625" height="78"></rect><g class="label" style="" transform="translate(-49.1328125, -24)"><rect></rect><foreignobject width="98.265625" height="48">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
strings + nulls<br>normal C#
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-B-1" transform="translate(298.7421875, 111)"><rect class="basic label-container" style="" x="-82.4765625" y="-39" width="164.953125" height="78"></rect><g class="label" style="" transform="translate(-52.4765625, -24)"><rect></rect><foreignobject width="104.953125" height="48">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
sealed records<br>idiomatic C#
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-C-3" transform="translate(534.1484375, 111)"><rect class="basic label-container" style="" x="-102.9296875" y="-39" width="205.859375" height="78"></rect><g class="label" style="" transform="translate(-72.9296875, -24)"><rect></rect><foreignobject width="145.859375" height="48">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
discriminated unions<br>F#
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-D-5" transform="translate(782.65625, 47)"><rect class="basic label-container" style="" x="-95.578125" y="-39" width="191.15625" height="78"></rect><g class="label" style="" transform="translate(-65.578125, -24)"><rect></rect><foreignobject width="131.15625" height="48">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
ADTs + NonEmpty<br>Haskell
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-E-7" transform="translate(782.65625, 175)"><rect class="basic label-container" style="" x="-80.9140625" y="-39" width="161.828125" height="78"></rect><g class="label" style="" transform="translate(-50.9140625, -24)"><rect></rect><foreignobject width="101.828125" height="48">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
enum + match<br>Rust
</p>
</span>
</div>
</foreignobject></g></g></g></g></g>
</svg>
</div>

| Engine | Shape | "Accepted with FatalMessage populated" |
|---|---|---|
| normal C# | one flat class + status string | structurally legal |
| idiomatic C# | sealed-record DU, private ctor | impossible to construct |
| F# | native DU | impossible to construct |
| Haskell | ADT + NonEmpty | impossible; non-empty enforced at type level |
| Rust | enum + newtype | impossible; exhaustive match enforced |

The thing to notice is **how much less the higher-rung types need to
say to mean the same thing**. The normal-C# type is 18 lines of
fields. The F# type is six lines. They model the same six outcomes.
The difference is who pays for the invariants — the compiler, or
every consumer at every callsite.

See [chapter 2 of the journey](../part1/02-normal-csharp-tour.ipynb)
for the narrative version of why this matters, and [chapter
04](../part1/04-idiomatic-csharp.ipynb) for the moment the type model
flips.